# TFM Tenerife — BERTopic MODELO A ("visión general")

Se entrena aquí (y no en local) porque `hdbscan` no se puede importar en la máquina de desarrollo -- Windows Smart App Control bloquea su librería nativa (ver `analytics/contexto.md`). Unos 3.100 documentos, entrena en segundos incluso en CPU, pero como ya estamos en Colab por el Modelo B, más fácil correrlo aquí también con GPU.

**Pasos**: ejecuta todas las celdas, sube `general_corpus.csv` cuando lo pida. Al final se descargan dos archivos: `general_topics_results.csv` (pásaselo a Claude, `analytics/topics/import_geo_results.py` vale para este archivo también) y `bertopic_model_general.zip` (el modelo entrenado — guárdalo por si hay que clasificar comentarios nuevos más adelante sin reentrenar desde cero).

In [ ]:
!pip install -q bertopic sentence-transformers

import torch
print('GPU disponible:', torch.cuda.is_available())

In [ ]:
from google.colab import files
print('Sube general_corpus.csv (generado por analytics/topics/export_general_corpus.py):')
uploaded = files.upload()

In [ ]:
import pandas as pd

nombre_csv = list(uploaded.keys())[0]
df = pd.read_csv(nombre_csv)
print(f'{len(df)} comentarios cargados.')
textos = df['text'].tolist()

In [ ]:
EMBEDDING_MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
MODEL_NAME = f'BERTopic+{EMBEDDING_MODEL_NAME} (modelo A: general)'
MIN_TOPIC_SIZE = 15

# Misma lista que analytics/topics/topic_modeling.py -- sin ella las etiquetas
# de tema salen "de, que, la, el" (BERTopic no trae stopwords en espanol).
SPANISH_STOPWORDS = {
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por',
    'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'mas', 'más',
    'pero', 'sus', 'le', 'ya', 'o', 'este', 'si', 'sí', 'porque', 'esta',
    'entre', 'cuando', 'muy', 'sin', 'sobre', 'tambien', 'también', 'me',
    'hasta', 'hay', 'donde', 'dónde', 'quien', 'quién', 'desde', 'todo',
    'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese',
    'eso', 'ante', 'ellos', 'e', 'esto', 'mi', 'antes', 'algunos', 'que',
    'unos', 'yo', 'otro', 'otras', 'otra', 'el', 'tanto', 'esa', 'estos',
    'mucho', 'quienes', 'nada', 'muchos', 'cual', 'cuál', 'poco', 'ella',
    'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tu', 'tú',
    'te', 'ti', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os',
    'mio', 'mío', 'mia', 'mía', 'tuyo', 'tuya', 'suyo', 'suya', 'es', 'soy',
    'eres', 'somos', 'sois', 'son', 'esté', 'esta', 'estan', 'están', 'fue',
    'ser', 'voy', 'vamos', 'va', 'van', 'puede', 'pueden', 'hace', 'hacer',
    'gracias', 'hola', 'saludos', 'pues', 'asi', 'así', 'aqui', 'aquí',
    'alli', 'allí', 'ahi', 'ahí',
}

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=5, metric='euclidean', cluster_selection_method='leaf', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS | SPANISH_STOPWORDS), ngram_range=(1, 2), min_df=2)
topic_model = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    min_topic_size=MIN_TOPIC_SIZE,
    language='multilingual',
    calculate_probabilities=False,
    verbose=True,
)

In [ ]:
topics, probs = topic_model.fit_transform(textos)

was_outlier = [t == -1 for t in topics]
n_before = sum(was_outlier)
topics = topic_model.reduce_outliers(textos, topics, strategy='c-tf-idf')
print(f'Outliers: {n_before} -> {topics.count(-1)}.')
probs = [None if wo else p for wo, p in zip(was_outlier, probs)]

topic_model.update_topics(
    textos, topics=topics,
    vectorizer_model=topic_model.vectorizer_model,
    ctfidf_model=topic_model.ctfidf_model,
)

topic_info = {}
for row in topic_model.get_topic_info().itertuples():
    words = [w for w, _ in topic_model.get_topic(row.Topic)] if row.Topic != -1 else []
    topic_info[row.Topic] = (', '.join(words[:6]) if words else 'outlier / sin tema claro', row.Count)
for topic_id, (label, size) in sorted(topic_info.items(), key=lambda kv: -kv[1][1]):
    tag = 'outlier' if topic_id == -1 else f'#{topic_id}'
    print(f'  {tag:>8} ({size:>4} docs): {label}')

In [ ]:
resultados = []
for (source, source_id, text), topic_id, prob in zip(zip(df['source'], df['source_id'], df['text']), topics, probs):
    label, size = topic_info.get(topic_id, (None, None))
    resultados.append({
        'source': source, 'source_id': source_id, 'text': text,
        'topic_id': int(topic_id), 'topic_label': label, 'topic_size': size,
        'probability': float(prob) if prob is not None else None,
        'model_name': MODEL_NAME,
    })

pd.DataFrame(resultados).to_csv('general_topics_results.csv', index=False)
print(f'{len(resultados)} filas guardadas.')

topic_model.save('bertopic_model_general', serialization='safetensors', save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_NAME)
!zip -rq bertopic_model_general.zip bertopic_model_general
print('Modelo guardado en bertopic_model_general.zip')

files.download('general_topics_results.csv')
files.download('bertopic_model_general.zip')